In [ ]:
import csv
import hashlib
import json
import os
import random
import re
import time
from datetime import datetime, timezone
from urllib.parse import urljoin
import html as _html

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.bhphotovideo.com"

CATEGORY_URLS = {
    "laptop": "https://www.bhphotovideo.com/c/buy/laptops/ci/18818",
}

OUT_DIR        = "./bh_output"
OUT_CSV        = os.path.join(OUT_DIR, "laptops_bh.csv")
CHECKPOINT     = os.path.join(OUT_DIR, "scraped_urls.txt")
URL_LIST_CACHE = os.path.join(OUT_DIR, "product_urls.txt")

MAX_PAGES_PER_CATEGORY = 40   
LIMIT                  = None
RESET_RUN              = True

DELAY_SECONDS  = 2.0
DELAY_JITTER   = 3.0
REQUEST_TIMEOUT = 300
MAX_RETRIES     = 5

MAX_CONSECUTIVE_BLOCKS = 5
BLOCK_BACKOFF_SECONDS  = 120

BLOCK_MARKERS = [
    "pardon our interruption", "access denied", "are you a human",
    "captcha", "request unsuccessful", "unusual traffic", "bot detection",
    "reference id", "akamai",
]

FIELD_MAP = {
    "cpu":     ["processor", "cpu type", "processor name", "cpu"],
    "ram":     ["installed ram", "memory", "ram", "system memory", "total ram"],
    "storage": ["total capacity", "ssd", "hard drive", "storage capacity", "storage", "hdd"],
    "gpu":     ["gpu", "graphics", "video card", "graphics card"],
    "display": ["display size", "screen size", "display", "screen"],
    "battery": ["capacity", "battery life", "battery"],
}

FIELDNAMES = [
    "title", "price", "price_original", "price_is_suspicious", "url", "sku_bh", "sku_mfr",
    "cpu", "ram", "storage", "gpu", "display", "battery",
    "rating", "review_count", "category", "description",
    "document_text", "reviews_json", "raw_specs_json",
    "field_provenance_json", "scraped_at", "html_sha256",
]

FIRECRAWL_KEYS = [
]
current_key_idx = 0

class BlockedError(Exception):
    pass

def build_session():
    return requests.Session()

def polite_sleep():
    time.sleep(DELAY_SECONDS + random.uniform(0, DELAY_JITTER))

def clean_text(s):
    if not s:
        return ""
    return re.sub(r"\s+", " ", s).strip()

def parse_price(value):
    if value is None:
        return ""
    text = str(value)
    text = re.sub(r"\s+", "", text)
    text = text.replace(",", "")
    match = re.search(r"(\d+(?:\.\d{1,2})?)", text)
    if not match:
        return ""
    try:
        return f"{float(match.group(1)):.2f}"
    except ValueError:
        return match.group(1)

def is_valid_price(price_str):
    try:
        return bool(price_str) and float(price_str) > 1.0
    except (ValueError, TypeError):
        return False

def looks_blocked(soup, resp_text):
    has_structure = bool(
        soup.select_one('[data-selenium="miniProductPage"]')
        or soup.select_one('h1[data-selenium="productTitle"]')
        or soup.select_one("meta[property='og:title']")
    )
    if has_structure:
        return False

    visible_text = soup.get_text(" ").lower()
    if any(marker in visible_text for marker in BLOCK_MARKERS):
        return True
    if len(resp_text) < 20000:
        return True
    return False

def get_soup(target_url, session, referer=None):
    global current_key_idx

    if not FIRECRAWL_KEYS:
        raise BlockedError("No Firecrawl API keys configured.")

    total_keys = len(FIRECRAWL_KEYS)


    network_attempts = 0

    full_rotations = 0

    while network_attempts < MAX_RETRIES:

        api_key = FIRECRAWL_KEYS[current_key_idx]

        headers = {
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        }

        payload = {
            "url": target_url,
            "formats": ["html"],
        }

        try:
            print(
                f"  [request] key {current_key_idx + 1}/{total_keys} "
                f"(attempt {network_attempts + 1}/{MAX_RETRIES})"
            )

            resp = session.post(
                "https://api.firecrawl.dev/v1/scrape",
                json=payload,
                headers=headers,
                timeout=(30, REQUEST_TIMEOUT),
            )


            if resp.status_code == 200:
                data = resp.json()

                if data.get("success") and "html" in data.get("data", {}):
                    html_content = data["data"]["html"]
                    soup = BeautifulSoup(html_content, "lxml")

                    if looks_blocked(soup, html_content):
                        raise BlockedError(
                            f"Block page detected inside Firecrawl "
                            f"payload at {target_url}"
                        )

                    return soup, html_content

                print(
                    "  [warn] Firecrawl returned HTTP 200 but "
                    "no usable HTML."
                )

                network_attempts += 1


            elif resp.status_code in [401, 402, 403, 429]:

                old_key = current_key_idx


                current_key_idx = (
                    current_key_idx + 1
                ) % total_keys

                print(
                    f"  [warn] Key {old_key + 1} failed "
                    f"with HTTP {resp.status_code}."
                )

                print(
                    f"  [rotate] Key {old_key + 1} -> "
                    f"Key {current_key_idx + 1}"
                )

           
                if current_key_idx == 0:
                    full_rotations += 1

                    cooldown = random.uniform(
                        BLOCK_BACKOFF_SECONDS,
                        BLOCK_BACKOFF_SECONDS * 1.75
                    )

                    print(
                        f"  [cooldown] Completed key rotation "
                        f"#{full_rotations}. "
                        f"Sleeping {cooldown:.1f}s..."
                    )

                    time.sleep(cooldown)

                else:
                    delay = random.uniform(4.0, 10.0)

                    print(
                        f"  [delay] Waiting {delay:.1f}s "
                        f"before next key..."
                    )

                    time.sleep(delay)

                continue


            elif resp.status_code in [
                408, 409, 425, 500, 502, 503, 504
            ]:

                network_attempts += 1

                base = min(
                    10 * (2 ** (network_attempts - 1)),
                    120
                )

                delay = random.uniform(
                    base * 0.75,
                    base * 1.25
                )

                print(
                    f"  [retry] HTTP {resp.status_code}. "
                    f"Retrying in {delay:.1f}s..."
                )

                time.sleep(delay)
                continue

   
            else:
                network_attempts += 1

                print(
                    f"  [warn] Firecrawl HTTP "
                    f"{resp.status_code} "
                    f"(attempt {network_attempts}/{MAX_RETRIES})"
                )

                if network_attempts < MAX_RETRIES:
                    base = min(
                        5 * (2 ** (network_attempts - 1)),
                        60
                    )

                    delay = random.uniform(
                        base * 0.75,
                        base * 1.25
                    )

                    print(
                        f"  [retry] Waiting {delay:.1f}s..."
                    )

                    time.sleep(delay)

                continue

        # -------------------------------------------------------------
        # NETWORK ERROR
        # -------------------------------------------------------------
        except requests.Timeout as e:
            network_attempts += 1

            base = min(
                10 * (2 ** (network_attempts - 1)),
                120
            )

            delay = random.uniform(
                base * 0.75,
                base * 1.25
            )

            print(
                f"  [timeout] {e} "
                f"(attempt {network_attempts}/{MAX_RETRIES})"
            )

            if network_attempts < MAX_RETRIES:
                print(
                    f"  [retry] Waiting {delay:.1f}s..."
                )
                time.sleep(delay)

        except requests.ConnectionError as e:
            network_attempts += 1

            base = min(
                10 * (2 ** (network_attempts - 1)),
                120
            )

            delay = random.uniform(
                base * 0.75,
                base * 1.25
            )

            print(
                f"  [connection] {e} "
                f"(attempt {network_attempts}/{MAX_RETRIES})"
            )

            if network_attempts < MAX_RETRIES:
                print(
                    f"  [retry] Waiting {delay:.1f}s..."
                )
                time.sleep(delay)

        except requests.RequestException as e:
            network_attempts += 1

            base = min(
                5 * (2 ** (network_attempts - 1)),
                60
            )

            delay = random.uniform(
                base * 0.75,
                base * 1.25
            )

            print(
                f"  [network error] {e} "
                f"(attempt {network_attempts}/{MAX_RETRIES})"
            )

            if network_attempts < MAX_RETRIES:
                print(
                    f"  [retry] Waiting {delay:.1f}s..."
                )
                time.sleep(delay)

    print(
        f"  [failed] Exhausted {MAX_RETRIES} "
        f"network attempts for {target_url}"
    )

    return None, None

def bh_page_url(search_url, page):
    if page <= 1:
        return search_url
    base = search_url.rstrip("/")
    return f"{base}/pn/{page}"

def extract_listing_card_data(card, base_url):
    link = card.select_one('a[data-selenium="miniProductPageProductNameLink"]')
    if not link or not link.get("href"):
        return None
    url = urljoin(base_url, link["href"].split("?")[0])

    name_tag = card.select_one('span[data-selenium="miniProductPageProductName"]')
    title = clean_text(name_tag.get_text()) if name_tag else clean_text(link.get_text())

    price = ""
    p1 = card.select_one('span[data-selenium="uppedDecimalPriceFirst"]')
    p2 = card.select_one('sup[data-selenium="uppedDecimalPriceSecond"]')
    if p1:
        dollars = re.sub(r"[^\d]", "", p1.get_text())
        cents = re.sub(r"[^\d]", "", p2.get_text()) if p2 else "00"
        cents = (cents + "00")[:2]
        if dollars:
            price = f"{dollars}.{cents}"

    price_original = ""
    strike = card.select_one('del[data-selenium="strikethroughPrice"]')
    if strike:
        price_original = parse_price(strike.get_text())

    sku_bh, sku_mfr = "", ""
    sku_tag = card.select_one('div[data-selenium="miniProductPageProductSkuInfo"]')
    if sku_tag:
        sku_text = clean_text(sku_tag.get_text())
        m_bh = re.search(r"BH\s*#\s*(\S+)", sku_text)
        m_mfr = re.search(r"MFR\s*#\s*(\S+)", sku_text)
        sku_bh = m_bh.group(1) if m_bh else ""
        sku_mfr = m_mfr.group(1) if m_mfr else ""

    bullets = card.select('li[data-selenium="miniProductPageSellingPointsListItem"]')
    key_features = [clean_text(b.get_text()) for b in bullets if clean_text(b.get_text())]

    rating_container = card.select_one('div[data-selenium="ratingContainer"]')
    rating = ""
    if rating_container:
        stars = rating_container.select("svg")
        full = sum(1 for s in stars if s.find("use") and s.find("use").get("href") == "#StarIcon")
        half = sum(1 for s in stars if s.find("use") and s.find("use").get("href") == "#StarHalfIcon")
        if stars:
            rating = str(full + 0.5 * half)

    review_count = ""
    rc_tag = card.select_one('span[data-selenium="miniProductPageProductReviews"]')
    if rc_tag:
        m = re.search(r"\d+", rc_tag.get_text())
        review_count = m.group(0) if m else ""

    return {
        "url": url,
        "title": title,
        "price": price,
        "price_original": price_original,
        "sku_bh": sku_bh,
        "sku_mfr": sku_mfr,
        "key_features": key_features,
        "rating": rating,
        "review_count": review_count,
    }

def collect_product_urls(search_url, max_pages, session):
    urls, seen = [], set()
    listing_data = {}

    for page in range(1, max_pages + 1):
        page_url = bh_page_url(search_url, page)
        print(f"[listing] page {page}: {page_url}")

        try:
            soup, _ = get_soup(page_url, session, referer=search_url)
        except BlockedError:
            print("  [blocked] listing page blocked — stopping early")
            break
        if soup is None:
            print("  [warn] could not load listing page — stopping")
            break

        cards = soup.select('div[data-selenium="miniProductPage"]')
        if not cards:
            print("  [warn] no product cards found — last page or selector stale")
            break

        new_this_page = 0
        for card in cards:
            data = extract_listing_card_data(card, BASE_URL)
            if not data:
                continue
            full = data["url"]
            if full not in seen:
                seen.add(full)
                urls.append(full)
                listing_data[full] = data
                new_this_page += 1

        print(f"  found {len(cards)} cards, {new_this_page} new (total {len(urls)})")
        if new_this_page == 0:
            break
        polite_sleep()

    return urls, listing_data

def collect_all_urls(session):
    all_urls, seen = [], set()
    all_listing_data = {}
    for category, url in CATEGORY_URLS.items():
        print(f"\n=== Collecting URLs for: {category} ===")
        urls, listing_data = collect_product_urls(url, MAX_PAGES_PER_CATEGORY, session)
        all_listing_data.update(listing_data)
        added = 0
        for u in urls:
            if u not in seen:
                seen.add(u)
                all_urls.append(u)
                added += 1
        print(f"[{category}] {len(urls)} collected, {added} new unique")
    return all_urls, all_listing_data

def backup_file(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d-%H%M%S")
        bak = f"{path}.bak-{ts}"
        os.replace(path, bak)
        print(f"[reset] backed up {path} -> {bak}")

def load_urls_from_file(path):
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    urls, seen = [], set()
    for chunk in re.split(r"(?=https?://)", text):
        chunk = chunk.strip()
        if not chunk.startswith("http"):
            continue
        url = chunk.split()[0].strip().rstrip(",").split("?")[0]
        if url and url not in seen:
            seen.add(url)
            urls.append(url)
    return urls

def load_checkpoint():
    return set(load_urls_from_file(CHECKPOINT))

def append_checkpoint(url):
    with open(CHECKPOINT, "a", encoding="utf-8") as f:
        f.write(url + "\n")

def extract_jsonld_product(soup):
    out = {
        "price": "", "sku_mfr": "", "sku_bh": "", "description": "",
        "specs": {}, "rating": "", "review_count": "", "reviews": [],
    }
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        items = data if isinstance(data, list) else [data]
        for item in items:
            if not isinstance(item, dict) or item.get("@type") != "Product":
                continue

            offers = item.get("offers", {})
            if isinstance(offers, list):
                offers = offers[0] if offers else {}
            if isinstance(offers, dict):
                p = parse_price(offers.get("price"))
                if is_valid_price(p):
                    out["price"] = p

            if item.get("sku"):
                out["sku_bh"] = clean_text(str(item["sku"]))
            if item.get("mpn"):
                out["sku_mfr"] = clean_text(str(item["mpn"]))

            if item.get("description"):
                out["description"] = clean_text(
                    BeautifulSoup(_html.unescape(item["description"]), "lxml").get_text(" ")
                )

            agg = item.get("aggregateRating")
            if isinstance(agg, dict):
                if agg.get("ratingValue") is not None:
                    out["rating"] = str(agg["ratingValue"])
                if agg.get("reviewCount") is not None:
                    out["review_count"] = str(agg["reviewCount"])

            for rv in item.get("review", []) or []:
                if not isinstance(rv, dict):
                    continue
                body = clean_text(_html.unescape(rv.get("reviewBody", "")))
                if body:
                    out["reviews"].append(body)

            specs_block = item.get("additionalProperty")
            spec_list = specs_block.get("value", []) if isinstance(specs_block, dict) else []
            for s in spec_list:
                if not (isinstance(s, dict) and s.get("name") and s.get("value") is not None):
                    continue
                key = clean_text(_html.unescape(str(s["name"])))
                raw_val = s["value"]
                if isinstance(raw_val, list):
                    val = clean_text("; ".join(_html.unescape(str(v)) for v in raw_val))
                else:
                    val = clean_text(_html.unescape(str(raw_val)))
                if key and val and key not in out["specs"]:
                    out["specs"][key] = val

            return out 
    return out

def extract_embedded_price(raw_html):
    if not raw_html:
        return ""
    for pat in [r'"salePrice"\s*:\s*([\d.]+)',
                r'"currentPrice"\s*:\s*([\d.]+)',
                r'"finalPrice"\s*:\s*([\d.]+)',
                r'"price"\s*:\s*"?([\d.]+)"?']:
        m = re.search(pat, raw_html)
        if m:
            price = parse_price(m.group(1))
            if is_valid_price(price):
                return price
    return ""

def extract_original_price_dom(soup):
    tag = soup.select_one('del[data-selenium="strikethroughPrice"]')
    if tag:
        return parse_price(tag.get_text())
    return ""

def extract_specs_dom(soup):
    rows = soup.select('tr[data-selenium="specsItemGroupTableRow"]')
    specs = {}
    for row in rows:
        lbl = row.select_one('td[data-selenium="specsItemGroupTableColumnLabel"]')
        val = row.select_one('td[data-selenium="specsItemGroupTableColumnValue"]')
        if lbl and val:
            key = clean_text(lbl.get_text())
            v = clean_text(val.get_text(" "))
            if key and v:
                specs[key] = v
    return specs

def is_bad_spec_match(field, candidate, raw_key):
    key = raw_key.lower()
    if field == "ram" and candidate in ("memory", "ram"):
        return any(w in key for w in
                   ["video", "vram", "max", "supported", "slot", "speed",
                    "type", "card", "configuration"])
    if field == "display" and candidate in ("display", "screen"):
        return any(w in key for w in
                   ["type", "resolution", "touch", "aspect", "brightness",
                    "color", "panel", "refresh"])
    if field == "storage" and candidate == "ssd":
        return any(w in key for w in
                   ["interface", "type", "slot", "read", "write", "protocol"])
    if field == "battery" and candidate == "capacity":
        return any(w in key for w in ["total", "storage", "ram", "l3", "cache"])
    return False

def extract_title_fallbacks(title):
    if not title:
        return {}
    t = title.lower()
    result = {}

    for pat in [r"(\d{1,3})\s*gb\s*(?:lpddr\d|ddr\d)",
                r"(\d{1,3})\s*gb\s+(?:ram|memory)"]:
        m = re.search(pat, t)
        if m:
            result["ram"] = f"{m.group(1)}GB"
            break

    m = re.search(r"(\d+(?:\.\d+)?)\s*tb\s*(?:ssd|nvme|pcie|storage)", t)
    if m:
        amt = float(m.group(1))
        result["storage"] = f"{int(amt)}TB" if amt == int(amt) else f"{amt}TB"
    else:
        m = re.search(r"(\d{2,4})\s*gb\s*(?:ssd|nvme|pcie|emmc|storage)", t)
        if m:
            result["storage"] = f"{m.group(1)}GB"

    for pat in [r"intel\s+core\s+ultra\s+\d\s+\d{3}[a-z]{0,3}",
                r"intel\s+core\s+i\d-\d{4}[a-z]{0,3}",
                r"intel\s+core\s+\d\s+\d{2,3}[a-z]{0,3}",
                r"intel\s+core\s+i\d",
                r"intel\s+processor\s+n\d{3}",
                r"amd\s+ryzen\s+ai\s+\d\s+\d{3}",
                r"amd\s+ryzen\s+\d\s+\d{4}[a-z]{0,3}",
                r"amd\s+ryzen\s+\d",
                r"apple\s+m\d(?:\s+pro|\s+max)?",
                r"qualcomm\s+snapdragon\s+x\s+(?:elite|plus)"]:
        m = re.search(pat, t)
        if m:
            result["cpu"] = m.group(0).title()
            break

    for pat in [r"geforce\s+rtx\s+\d{4}\s*(?:ti|super)?(?:\s+laptop\s+gpu)?",
                r"gtx\s+\d{4}\s*(?:ti|super)?",
                r"amd\s+radeon\s+rx\s+\d{4}[a-z]*",
                r"intel\s+arc\s+\d{3}[a-z]*",
                r"radeon\s+\d{3}[a-z]*",
                r"intel\s+iris\s+xe\s+graphics",
                r"amd\s+radeon\s+graphics",
                r"qualcomm\s+adreno(?:\s+gpu)?",
                r"adreno(?:\s+gpu)?"]:
        m = re.search(pat, t)
        if m:
            result["gpu"] = m.group(0).title()
            break

    m = re.search(r'(\d{1,2}(?:\.\d)?)\s*(?:"|in\b|inch)', t)
    if m:
        result["display"] = f'{m.group(1)}"'

    m = re.search(r"up to\s+(\d+)\s*hours?", t)
    if m:
        result["battery"] = f"Up to {m.group(1)} Hours"

    return result

def map_specs(specs, title=""):
    mapped, provenance = {}, {}
    lower_specs = {k.lower().strip(): v for k, v in specs.items()}
    title_fb = extract_title_fallbacks(title)

    for field, candidates in FIELD_MAP.items():
        value, matched_cand, matched_key = "", None, None

        for cand in candidates:
            if cand in lower_specs:
                value = lower_specs[cand]
                matched_cand, matched_key = cand, cand
                break

        if not value:
            for cand in candidates:
                for raw_key, raw_val in lower_specs.items():
                    if cand in raw_key and not is_bad_spec_match(field, cand, raw_key):
                        value = raw_val
                        matched_cand, matched_key = cand, raw_key
                        break
                if value:
                    break

        if not value and field in title_fb:
            value = title_fb[field]
            matched_cand, matched_key = "title_regex", "title"

        mapped[field] = value
        provenance[field] = (
            {"candidate": matched_cand, "raw_key": matched_key}
            if value else None
        )

    return mapped, provenance

def extract_description_dom(soup):
    tag = soup.select_one('div[data-selenium="sellingPointsOverviewDescription"]')
    if tag:
        text = clean_text(tag.get_text(" "))
        if len(text) > 20:
            return text
    meta = soup.select_one("meta[name='description'], meta[property='og:description']")
    if meta and meta.get("content"):
        return clean_text(meta["content"])
    return ""

def extract_category(soup):
    cat_link = soup.select_one('a[data-selenium="overviewExpertPicksCategoryLink"]')
    if cat_link:
        text = clean_text(cat_link.get_text())
        if text:
            return text

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue
        if isinstance(data, dict) and data.get("@type") == "BreadcrumbList":
            items = sorted(data.get("itemListElement", []), key=lambda x: x.get("position", 0))
            names = [clean_text(i.get("item", {}).get("name", "")) for i in items]
            names = [n for n in names if n and n != "BH Photo"]
            if names:
                return " > ".join(names)

    crumbs = soup.select('a[data-selenium="linkCrumb"]')
    names = [clean_text(c.get_text()) for c in crumbs if clean_text(c.get_text()) != "Home"]
    if names:
        return " > ".join(names)
    return ""

def scrape_product_page(url, session, listing_fallback=None):
    listing_fallback = listing_fallback or {}
    soup, raw_html = get_soup(url, session, referer=BASE_URL)
    if soup is None:
        return None

    title_tag = soup.select_one('h1[data-selenium="productTitle"]')
    title = clean_text(title_tag.get_text()) if title_tag else ""
    if not title:
        og = soup.select_one("meta[property='og:title']")
        if og and og.get("content"):
            title = clean_text(og["content"])
    if not title:
        title = listing_fallback.get("title", "")

    jsonld = extract_jsonld_product(soup)

    price = jsonld["price"] or extract_embedded_price(raw_html) or listing_fallback.get("price", "")
    price_original = extract_original_price_dom(soup) or listing_fallback.get("price_original", "")

    try:
        price_float = float(price) if price else None
    except ValueError:
        price_float = None
    price_suspicious = "1" if (price_float is None or price_float < 100) else "0"

    specs = jsonld["specs"] or extract_specs_dom(soup)
    mapped, provenance = map_specs(specs, title)
    
    if listing_fallback.get("key_features"):
        bullet_fb = extract_title_fallbacks(" ".join(listing_fallback["key_features"]))
        for field, val in bullet_fb.items():
            if not mapped.get(field):
                mapped[field] = val
                provenance[field] = {"candidate": "listing_bullets", "raw_key": None}

    description = jsonld["description"] or extract_description_dom(soup)
    reviews = jsonld["reviews"]
    rating = jsonld["rating"] or listing_fallback.get("rating", "")
    review_count = jsonld["review_count"] or listing_fallback.get("review_count", "")
    sku_bh = jsonld["sku_bh"] or listing_fallback.get("sku_bh", "")
    sku_mfr = jsonld["sku_mfr"] or listing_fallback.get("sku_mfr", "")
    category = extract_category(soup)

    parts = []
    if title:
        parts.append(f"Product: {title}")
    if category:
        parts.append(f"Category: {category}")
    if price:
        parts.append(f"Price: ${price}")
    if specs:
        spec_str = "; ".join(f"{k}: {v}" for k, v in specs.items())
        parts.append(f"Specifications: {spec_str}")
    elif listing_fallback.get("key_features"):
        parts.append("Key Features: " + " | ".join(listing_fallback["key_features"]))
    if description:
        parts.append(f"Description: {description}")
    if reviews:
        parts.append("Customer Reviews: " + " | ".join(reviews[:5]))
    document_text = "\n".join(parts)

    scraped_at = datetime.now(timezone.utc).isoformat()
    html_sha256 = hashlib.sha256(raw_html.encode("utf-8")).hexdigest() if raw_html else ""

    return {
        "title": title,
        "price": price,
        "price_original": price_original,
        "price_is_suspicious": price_suspicious,
        "url": url,
        "sku_bh": sku_bh,
        "sku_mfr": sku_mfr,
        "cpu": mapped["cpu"],
        "ram": mapped["ram"],
        "storage": mapped["storage"],
        "gpu": mapped["gpu"],
        "display": mapped["display"],
        "battery": mapped["battery"],
        "rating": rating,
        "review_count": review_count,
        "category": category,
        "description": description,
        "document_text": document_text,
        "reviews_json": json.dumps(reviews, ensure_ascii=False),
        "raw_specs_json": json.dumps(specs, ensure_ascii=False),
        "field_provenance_json": json.dumps(provenance, ensure_ascii=False),
        "scraped_at": scraped_at,
        "html_sha256": html_sha256,
    }

def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    if RESET_RUN:
        print("[reset] Backing up old output files...")
        backup_file(OUT_CSV)
        backup_file(CHECKPOINT)

    session = build_session()
    
    print("\n=== Step 1: collecting / loading product URLs ===")
    listing_data = {}
    if os.path.exists(URL_LIST_CACHE):
        product_urls = load_urls_from_file(URL_LIST_CACHE)
        print(f"Loaded {len(product_urls)} URLs from cached {URL_LIST_CACHE}")
    else:
        product_urls, listing_data = collect_all_urls(session)
        with open(URL_LIST_CACHE, "w", encoding="utf-8") as f:
            f.write("\n".join(product_urls))
        print(f"Total product URLs collected: {len(product_urls)}")

    if not product_urls:
        print("No product URLs found. Exiting.")
        return

    already_done = load_checkpoint()
    remaining = [u for u in product_urls
                 if urljoin(BASE_URL, u) not in already_done]
    print(f"{len(already_done)} already scraped, {len(remaining)} remaining")

    if LIMIT:
        remaining = remaining[:LIMIT]
        print(f"Limiting this run to {len(remaining)} products")

    print("\n=== Step 2: scraping product pages ===")
    consecutive_blocks = 0
    scraped_this_run = 0

    with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if f.tell() == 0:
            writer.writeheader()

        for i, url in enumerate(remaining, 1):
            full_url = urljoin(BASE_URL, url)
            print(f"[{i}/{len(remaining)}] {full_url}")

            try:
                row = scrape_product_page(full_url, session, listing_data.get(full_url))
            except BlockedError as e:
                consecutive_blocks += 1
                print(f"  [BLOCKED] {e} "
                      f"({consecutive_blocks}/{MAX_CONSECUTIVE_BLOCKS})")
                if consecutive_blocks >= MAX_CONSECUTIVE_BLOCKS:
                    print(f"\n[stop] {MAX_CONSECUTIVE_BLOCKS} consecutive blocks. "
                          f"{scraped_this_run} saved. Re-run later to resume.")
                    break
                time.sleep(BLOCK_BACKOFF_SECONDS)
                continue

            if row is None:
                print("  [skip] page load failed (non-block)")
                polite_sleep()
                continue

            consecutive_blocks = 0
            writer.writerow(row)
            f.flush()
            append_checkpoint(full_url)
            scraped_this_run += 1
            polite_sleep()

    print(f"\nDone. {scraped_this_run} products scraped this run.")
    print(f"Output: {OUT_CSV}")
    print(f"Checkpoint: {CHECKPOINT}")

if __name__ == "__main__":
    main()